# 09 Photo features: clean the photos, measure them, and test which parts matter

**Goal:** turn each cut-edge photo into a short list of numbers (**features**), check that a simple model can read the defect from them, and run an **ablation study** to find out which features and cleaning steps actually matter.
The result is a table of *photo evidence* (predicted defect + confidence per photo) that the knowledge graph can link to each job in Phase 2.

**Why measure features instead of just showing the photo to an LLM?** Numbers can be checked, compared across jobs and cited ("dark band covers 46% of the edge"). A vision LLM is the later step (system 5).

**Approach, and the papers behind it**

| Step | What we do | Paper it comes from |
|---|---|---|
| Clean | fix file names, match photos to jobs, remove blurry photos, correct uneven lighting | Sundaravadivel 2026 (lighting normalisation for reflective metal); Cang 2026 (fixed lighting matters) |
| Hand features | measure what an inspector looks at: edge tilt, dark melt band, dross blobs, striation lines | Guerra 2026 (laser cutting: geometric + intensity image features predict dross and faults); Fan 2026 (describe each defect by readable attributes) |
| Feature selection | keep features that relate to a defect, drop ones that repeat each other | Guerra 2026 (relevance > 0.5, redundancy > 0.6) |
| Model | random forest + a prototype (nearest-average) classifier | Cang 2026 (prototype few-shot classification) |
| Learned features | raw pixels squeezed with PCA here; ResNet-18 on your laptop | Wu 2023 (CNN image features matched to graph entities) |
| Ablation | switch one part off at a time and see how much the score drops | standard practice (e.g. SAM-LLaVA ablates each module) |

**Inputs:** `datasets/quality/photos/`, `datasets/machine/job_log.csv`, and `data/processed/event_timeline.csv` from notebook 01 (for the inspector's label).
**Outputs:** `data/processed/photo_features.csv`, `data/processed/photo_evidence.csv`, `data/processed/photo_ablation.csv`, figures in `data/processed/figures/`.

If `scipy` or `sklearn` is missing, run `pip install scipy scikit-learn` once in the Anaconda Prompt.

## 0. Setup

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestCentroid
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance
from sklearn.base import BaseEstimator, TransformerMixin

DATA = Path("..") / "datasets"
OUT = Path("..") / "data" / "processed"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)
PHOTOS = DATA / "quality" / "photos"
CLASSES = ["ok", "dross", "incomplete_cut", "rough_cut", "bevel_deviation"]   # burnt_edge (1 job) is too rare to learn

## 1. Cleaning step 1: file names → job IDs
Photos should be called `JOB-00123_edge.jpg`, but phones and people rename things. We pull the job ID out of each name with a **regular expression** (a text pattern: `JOB-` followed by 5 digits), so `JOB-00066 edge (2).jpg` still counts.
Then we check that no job has two photos and that every file opens.

In [ ]:
files = sorted(PHOTOS.glob("*.jpg"))
photos = pd.DataFrame({"file": files})
photos["name"] = photos["file"].apply(lambda p: p.name)
photos["job_id"] = photos["name"].str.extract(r"(JOB-\d{5})", expand=False)
odd = photos[~photos["name"].str.fullmatch(r"JOB-\d{5}_edge\.jpg")]
print(len(photos), "photo files;", len(odd), "with an odd name (job ID still readable):")
print(odd["name"].tolist())
assert photos["job_id"].notna().all(), "a photo name with no job ID: look at it by hand"
assert not photos["job_id"].duplicated().any(), "two photos for one job: decide which to keep"

sizes = photos["file"].apply(lambda p: Image.open(p).size)
print("image sizes:", sizes.value_counts().to_dict())

## 2. Cleaning step 2: attach the inspector's label
The label (the "answer" the model learns) is what the inspector wrote, cleaned in notebook 01. A photo is only useful for training if its job has a usable label.

In [ ]:
tl = pd.read_csv(OUT / "event_timeline.csv")
insp = tl[tl["event_type"] == "inspection"][["job_id", "detail"]].copy()
insp["label"] = insp["detail"].str.split(" | ", regex=False).str[1]
photos = photos.merge(insp[["job_id", "label"]], on="job_id", how="left")

jobs = pd.read_csv(DATA / "machine" / "job_log.csv", sep=";", decimal=",").drop_duplicates()
photos = photos.merge(jobs[["JobID", "Material", "Thickness_mm", "BevelAngle_deg"]], left_on="job_id", right_on="JobID", how="left").drop(columns="JobID")
assert photos["Material"].notna().all(), "a photo whose job is not in the job log"

print("photos per inspector label:")
photos["label"].fillna("(not inspected)").value_counts()

### Look before you measure
Always look at a few photos of each class first. You should be able to say in words what each defect looks like; those words become the features in step 5.

In [ ]:
show = ["ok", "dross", "incomplete_cut", "rough_cut", "bevel_deviation", "unknown"]
fig, axes = plt.subplots(len(show), 4, figsize=(12, 2.0 * len(show)))
for r, lab in enumerate(show):
    pick = photos[photos["label"] == lab].sample(4, random_state=1)
    for c, (_, p) in enumerate(pick.iterrows()):
        axes[r, c].imshow(Image.open(p["file"]))
        axes[r, c].set_title(f"{lab}  {p['job_id']}  bevel {p['BevelAngle_deg']:g}°", fontsize=8)
        axes[r, c].axis("off")
plt.tight_layout(); plt.savefig(FIG / "photos_by_label.png", dpi=150); plt.show()

**What to notice**
- **ok:** a clean rectangle with fine, straight vertical lines (striations).
- **dross:** small dark blobs stuck to the *bottom* edge.
- **incomplete_cut:** a wide dark, bubbly band over the lower part of the edge.
- **rough_cut:** lines are closer together, darker and wavy.
- **bevel_deviation:** the edge is tilted *more or less than the program asked for*. A 22.5° bevel job is *supposed* to look tilted, so the photo alone can't tell; you also need the programmed bevel angle from the job log. Keep this in mind for the ablation.
- **unknown:** the inspector didn't decide, but several of these clearly show a defect. The model can suggest one later (step 9).

## 3. Cleaning step 3: find blurry photos
A blurry photo hides fine lines and blobs, so its features are wrong. A standard sharpness score is the **variance of the Laplacian**: the Laplacian highlights sudden brightness changes (edges); a sharp photo has many strong ones, so the variance is high, and a blurry photo has almost none.

In [ ]:
def sharpness(path):
    g = np.asarray(Image.open(path).convert("L"), dtype=np.float32)
    return float(ndimage.laplace(g).var())

photos["sharpness"] = photos["file"].apply(sharpness)
plt.figure(figsize=(7, 3))
plt.hist(np.log10(photos["sharpness"]), bins=50)
plt.axvline(np.log10(50), color="red", ls="--", label="threshold = 50")
plt.xlabel("log10(sharpness)"); plt.ylabel("photos"); plt.legend(); plt.title("Sharpness of every photo")
plt.tight_layout(); plt.savefig(FIG / "photo_sharpness.png", dpi=150); plt.show()

BLUR_LIMIT = 50   # read off the histogram: there is a clear gap between the blurry group and the rest
photos["blurry"] = photos["sharpness"] < BLUR_LIMIT
print(photos["blurry"].sum(), "blurry photos out of", len(photos))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 2.2))
for ax, (_, p) in zip(axes, photos[photos["blurry"]].head(4).iterrows()):
    ax.imshow(Image.open(p["file"])); ax.set_title(f"{p['job_id']} sharpness {p['sharpness']:.0f}", fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

These photos are **flagged, not deleted**: the file stays, but the model doesn't learn from it and the graph marks it "photo unreadable" (in real life: ask the inspector to retake it). The ablation in step 7 checks what happens if we keep them.

## 4. Cleaning step 4: correct the uneven lighting
Phone photos on a bench are brighter on one side. If we don't fix that, "dark" means something different on the left and right of the same photo. We measure the brightness of the plain background strip at the top and bottom of each photo, column by column, and divide it out (a simple *flat-field correction*). Sundaravadivel 2026 does the same job with CLAHE for shiny metal.

In [ ]:
def load_grey(path, correct_light=True):
    g = np.asarray(Image.open(path).convert("L"), dtype=np.float32)
    if correct_light:
        bg = np.concatenate([g[:20], g[-20:]]).mean(axis=0)   # background brightness per column
        bg = ndimage.uniform_filter1d(bg, 31)                 # smooth it so we remove the trend, not detail
        g = g / bg * bg.mean()
    return g

p = photos[photos["label"] == "ok"].iloc[0]["file"]
raw, fixed = load_grey(p, False), load_grey(p, True)
fig, axes = plt.subplots(1, 3, figsize=(13, 2.6))
axes[0].imshow(raw, cmap="gray", vmin=0, vmax=255); axes[0].set_title("before"); axes[0].axis("off")
axes[1].imshow(fixed, cmap="gray", vmin=0, vmax=255); axes[1].set_title("after"); axes[1].axis("off")
axes[2].plot(raw[:20].mean(axis=0), label="before"); axes[2].plot(fixed[:20].mean(axis=0), label="after")
axes[2].set_title("background brightness, left → right"); axes[2].legend()
plt.tight_layout(); plt.show()

## 5. Measure the features
Each feature answers one question an inspector would ask. All "dark" tests are **relative** to the photo's own typical brightness, so a dark steel and a bright aluminium edge are judged the same way.

| Group | Feature | Question it answers | Defect it should reveal |
|---|---|---|---|
| quality | `sharpness`, `brightness`, `light_gradient` | is the photo usable, how is it lit? | (none: a check) |
| geometry | `face_height_px` | how thick does the edge look? | (context) |
| geometry | `tilt_deg` | how tilted is the cut edge? | bevel |
| geometry | `tilt_off` | how far is the tilt from **other photos with the same programmed bevel angle**? | bevel_deviation |
| dark spots | `dark_frac_lower`, `dark_frac_upper`, `dark_band_height` | how much of the lower/upper edge is dark melt, and how high does it reach? | incomplete_cut |
| dark spots | `blob_count`, `blob_mean_size` | how many separate dark spots, how big? | dross (many small), incomplete_cut (large) |
| edges | `below_edge_dark`, `bottom_edge_ragged`, `top_edge_ragged` | is the bottom/top boundary clean or ragged? | dross, burnt_edge |
| striations | `stripes_per100px_*`, `stripe_contrast_*`, `stripe_straight_*` (top and lower half) | how close, how dark and how straight are the lines? | rough_cut |

`tilt_off` uses the job log, not just the photo. It uses **no labels** (it compares with *all* photos of that bevel angle, good or bad), so it doesn't leak the answer.

How the face is found: **Otsu's threshold** picks the grey level that best splits the photo into a dark background and a bright cut face. The face rows and its left/right boundary lines come from that split.

In [ ]:
def otsu(values):
    # Otsu's threshold: the grey level that best separates dark pixels from bright ones
    hist, edges = np.histogram(values, bins=256, range=(0, 256))
    p = hist / hist.sum()
    w, mu = np.cumsum(p), np.cumsum(p * np.arange(256))
    between = (mu[-1] * w - mu) ** 2 / (w * (1 - w) + 1e-12)
    return edges[np.argmax(between)]


def edge_features(path, correct_light=True):
    g = load_grey(path, correct_light)
    H, W = g.shape
    f = {"sharpness": float(ndimage.laplace(g).var())}

    # --- find the cut face (bright) on the background (dark) ---
    mask = ndimage.binary_opening(g > otsu(g), iterations=1)
    rows = np.where(mask[:, W // 4: 3 * W // 4].mean(axis=1) > 0.5)[0]
    if len(rows) < 10:
        return None                                   # no face found: leave this photo out
    top, bot = rows.min(), rows.max()
    face_h = bot - top + 1
    face, fmask = g[top:bot + 1], mask[top:bot + 1]
    med = float(np.median(face[fmask]))               # the face's typical brightness
    f["face_height_px"] = face_h
    f["brightness"] = med
    f["light_gradient"] = float(g[:20, -40:].mean() / (g[:20, :40].mean() + 1e-6))

    # --- tilt: fit a straight line to the left and right boundary of the face ---
    ks = np.arange(int(face_h * 0.1), int(face_h * 0.9))
    left = np.array([np.argmax(fmask[k]) for k in ks])
    right = np.array([W - 1 - np.argmax(fmask[k][::-1]) for k in ks])
    pl, pr = np.polyfit(ks, left, 1), np.polyfit(ks, right, 1)
    f["tilt_deg"] = float(np.degrees(np.arctan((pl[0] + pr[0]) / 2)))
    kk, cc = np.arange(face_h)[:, None], np.arange(W)[None, :]
    inside = (cc > np.polyval(pl, kk) + 6) & (cc < np.polyval(pr, kk) - 6)   # 6 px in from the sloped sides

    # --- dark spots inside the face ---
    dark = ((face < 0.6 * med) | ~fmask) & inside
    rowdark = dark.sum(axis=1) / inside.sum(axis=1).clip(1)                # dark share of each row
    f["dark_frac_lower"] = float(rowdark[int(face_h * 0.65):].mean())
    f["dark_frac_upper"] = float(rowdark[: int(face_h * 0.5)].mean())
    hit = np.where(rowdark > 0.08)[0]
    f["dark_band_height"] = float((face_h - hit.min()) / face_h) if len(hit) else 0.0
    lab, n = ndimage.label(dark[int(face_h * 0.5):])                        # separate blobs in the lower half
    sizes = ndimage.sum(np.ones_like(lab), lab, range(1, n + 1)) if n else np.array([])
    sizes = sizes[sizes >= 4]                                               # ignore specks of noise
    f["blob_count"] = int(len(sizes))
    f["blob_mean_size"] = float(sizes.mean()) if len(sizes) else 0.0

    # --- top and bottom boundaries: dross hangs below, burn-outs bite the top ---
    cols = np.arange(40, W - 40)
    f["below_edge_dark"] = float((g[bot - 3: min(H, bot + 15), 40: W - 40] < 0.6 * med).mean())
    last = [np.where(mask[:, c])[0].max() if mask[:, c].any() else bot for c in cols]
    first = [np.where(mask[:, c])[0].min() if mask[:, c].any() else top for c in cols]
    f["bottom_edge_ragged"] = float(np.std(last))
    f["top_edge_ragged"] = float(np.std(first))

    # --- striations: thin dark lines across each row ---
    def stripes(band):
        prof = band[:, 30:-30]
        hp = prof - ndimage.uniform_filter1d(prof, 9, axis=1)              # keep only fine detail
        mins = (hp[:, 1:-1] < hp[:, :-2]) & (hp[:, 1:-1] < hp[:, 2:]) & (hp[:, 1:-1] < -0.03 * med)
        per100 = mins.sum(axis=1).mean() / prof.shape[1] * 100              # lines per 100 px
        straight = np.nanmean([np.corrcoef(hp[k], hp[k + 4])[0, 1] for k in range(0, len(hp) - 4, 2)])
        return per100, hp.std() / med, float(straight)                     # straight lines -> rows look alike
    upper = face[int(face_h * 0.1): int(face_h * 0.45)]
    lower = face[int(face_h * 0.55): int(face_h * 0.9)]
    f["stripes_per100px_top"], f["stripe_contrast_top"], f["stripe_straight_top"] = stripes(upper)
    f["stripes_per100px_low"], f["stripe_contrast_low"], f["stripe_straight_low"] = stripes(lower)
    return f


def measure_all(correct_light=True):
    rows = []
    for _, p in photos.iterrows():
        f = edge_features(p["file"], correct_light)
        if f is not None:
            rows.append({"job_id": p["job_id"], **f})
    feats = pd.DataFrame(rows).merge(photos[["job_id", "label", "blurry", "BevelAngle_deg", "Material", "Thickness_mm"]], on="job_id")
    # tilt compared with all photos of the same programmed bevel angle (no labels used)
    feats["tilt_off"] = (feats["tilt_deg"] - feats.groupby("BevelAngle_deg")["tilt_deg"].transform("median")).abs()
    return feats.round(4)

feats = measure_all(correct_light=True)
print(len(feats), "photos measured (of", len(photos), ")")
feats.to_csv(OUT / "photo_features.csv", index=False)
feats.head()

### Does each feature separate the classes?
A **box plot** shows the middle 50% of values for each class as a box, with the median as a line. If the boxes of two classes don't overlap, that feature alone can tell them apart.

In [ ]:
GROUPS = {
    "quality":    ["sharpness", "brightness", "light_gradient"],
    "geometry":   ["face_height_px", "tilt_deg", "tilt_off"],
    "dark spots": ["dark_frac_lower", "dark_frac_upper", "dark_band_height", "blob_count", "blob_mean_size"],
    "edges":      ["below_edge_dark", "bottom_edge_ragged", "top_edge_ragged"],
    "striations": ["stripes_per100px_top", "stripe_contrast_top", "stripe_straight_top",
                   "stripes_per100px_low", "stripe_contrast_low", "stripe_straight_low"],
}
FEATURES = [c for g in GROUPS.values() for c in g]

train = feats[feats["label"].isin(CLASSES) & ~feats["blurry"]].reset_index(drop=True)
print("photos used for training:", len(train)); print(train["label"].value_counts().to_dict())

key = ["tilt_off", "dark_band_height", "blob_count", "bottom_edge_ragged", "stripe_contrast_top", "stripe_straight_low"]
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for ax, col in zip(axes.flat, key):
    ax.boxplot([train.loc[train["label"] == c, col] for c in CLASSES], showfliers=False)
    ax.set_xticks(range(1, len(CLASSES) + 1), [c.replace("_", "\n") for c in CLASSES], fontsize=8)
    ax.set_title(col)
plt.tight_layout(); plt.savefig(FIG / "photo_feature_boxplots.png", dpi=150); plt.show()

**How to read it** (from my run; check yours):
- `tilt_off` is high only for bevel_deviation.
- `dark_band_height` sits around half the edge for incomplete_cut, as intended.
- `stripe_contrast_top` stands out for rough_cut.
- `bottom_edge_ragged` is near zero for ok photos and clearly above zero for dross.

So each defect has at least one feature that "points" at it. One surprise worth writing up: rough_cut has `dark_band_height` = 1 and a high `blob_count`, because its striation lines are so dark that the "dark spot" test counts them as spots. A feature can measure something other than what you designed it for, and the box plot is how you catch that.

## 6. Feature selection (Guerra 2026 method)
Guerra et al. kept a feature if its correlation with the target was above **0.5** (*relevant*), then dropped one of any pair correlated above **0.6** with each other (*redundant*: they measure the same thing).
Our target has 5 classes, so for each feature we take its strongest correlation with any one class (as a yes/no column).

In [ ]:
onehot = pd.get_dummies(train["label"]).astype(float)
relevance = pd.Series({f: max(abs(train[f].corr(onehot[c])) for c in CLASSES) for f in FEATURES}).sort_values(ascending=False)
relevant = relevance[relevance > 0.5].index.tolist()

selected = []
for f in relevant:   # most relevant first; skip a feature if it repeats one we already kept
    if all(abs(train[f].corr(train[s])) <= 0.6 for s in selected):
        selected.append(f)

print("relevance (strongest |r| with any class):"); print(relevance.round(2).to_string())
print("\nrelevant (> 0.5):", relevant)
print("selected after removing redundancy (> 0.6):", selected)

plt.figure(figsize=(8, 6.5))
cm = train[FEATURES].corr()
plt.imshow(cm, cmap="RdBu_r", vmin=-1, vmax=1); plt.colorbar(label="correlation")
plt.xticks(range(len(FEATURES)), FEATURES, rotation=90, fontsize=7); plt.yticks(range(len(FEATURES)), FEATURES, fontsize=7)
plt.title("Which features measure the same thing?"); plt.tight_layout(); plt.savefig(FIG / "photo_feature_correlation.png", dpi=150); plt.show()

## 7. Train a model and score it fairly
**Random forest:** many small decision trees, each asking yes/no questions about the features ("is `blob_count` > 3?"), that vote. It needs no scaling and tells us which features it used.

**How we score it**
- **Cross-validation, 5 folds × 3 repeats:** split the photos into 5 parts, train on 4 and test on the 5th, rotate, then repeat with 3 different shuffles. Every photo is tested by a model that never saw it.
- **Macro F1:** the F1 score (a balance of "when it says dross, is it right?" and "does it find all the dross?") for each class, then averaged so every class counts the same. This matters because 62% of the photos are "ok": a model that always says ok would get 62% accuracy but a macro F1 near 0.15.
- `class_weight="balanced"` tells the forest that rare classes (bevel_deviation: 10 photos) matter as much as common ones.

In [ ]:
CV = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
forest = lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced", random_state=42, n_jobs=-1)

def score(model, X, y):
    r = cross_validate(model, X, y, cv=CV, scoring=["f1_macro", "accuracy"])
    return {"macro_F1": r["test_f1_macro"].mean(), "F1_sd": r["test_f1_macro"].std(), "accuracy": r["test_accuracy"].mean(), "photos": len(y)}

y = train["label"]
always_ok = (y == "ok").mean()
print(f"baseline 'always ok': accuracy {always_ok:.1%}")
full = score(forest(), train[FEATURES], y)
print(f"random forest, all features: macro F1 {full['macro_F1']:.3f} ± {full['F1_sd']:.3f}, accuracy {full['accuracy']:.1%}")

In [ ]:
pred = cross_val_predict(forest(), train[FEATURES], y, cv=StratifiedKFold(5, shuffle=True, random_state=42))
print(classification_report(y, pred, labels=CLASSES, digits=2))
ConfusionMatrixDisplay.from_predictions(y, pred, labels=CLASSES, xticks_rotation=45, cmap="Blues")
plt.title("Photo model: inspector label (rows) vs prediction (columns)")
plt.tight_layout(); plt.savefig(FIG / "photo_confusion.png", dpi=150); plt.show()

**How to read the confusion matrix:** each row is what the inspector said, each column is what the model said. Numbers on the diagonal are correct. A number off the diagonal is a mix-up, for example row `dross`, column `ok` = dross photos the model called ok.

## 8. Ablation study: switch one thing off at a time
An **ablation** removes one part of the method and re-scores it. If the score drops a lot, that part was doing real work; if it doesn't move, the part isn't needed. Every row below uses the same cross-validation as step 7, so the rows can be compared.

The rows are in four groups:
1. **Feature groups:** drop one group, or use one group alone.
2. **Cleaning:** no lighting correction; keep the blurry photos.
3. **Context:** photo only (no `tilt_off`, so no programmed bevel angle).
4. **Method:** Guerra-selected features only; prototype classifier (Cang 2026); raw pixels + PCA instead of hand features.

In [ ]:
results = {"FULL: all features, cleaned photos": full}

# 1. feature groups
for g, cols in GROUPS.items():
    results[f"without {g}"] = score(forest(), train[[c for c in FEATURES if c not in cols]], y)
for g, cols in GROUPS.items():
    results[f"only {g}"] = score(forest(), train[cols], y)

# 2. cleaning
feats_nolight = measure_all(correct_light=False)
t2 = feats_nolight[feats_nolight["label"].isin(CLASSES) & ~feats_nolight["blurry"]]
results["no lighting correction"] = score(forest(), t2[FEATURES], t2["label"])
t3 = feats[feats["label"].isin(CLASSES)]
results["keep blurry photos"] = score(forest(), t3[FEATURES], t3["label"])

# 3. context
results["photo only (no tilt_off)"] = score(forest(), train[[c for c in FEATURES if c != "tilt_off"]], y)

# 4. method
# the selection itself looks at labels, so it must be re-done inside every training fold; otherwise the test photos
# helped choose the features and the score is optimistic (a classic leak). GuerraSelect does that inside the pipeline.
class GuerraSelect(BaseEstimator, TransformerMixin):
    def fit(self, X, y):
        X = pd.DataFrame(X, columns=FEATURES)
        oh = pd.get_dummies(pd.Series(y)).astype(float)
        rel = pd.Series({f: max(abs(X[f].corr(pd.Series(oh[c].values, index=X.index))) for c in oh.columns) for f in FEATURES}).fillna(0)
        self.keep_ = []
        for f in rel[rel > 0.5].sort_values(ascending=False).index:
            if all(abs(X[f].corr(X[s])) <= 0.6 for s in self.keep_):
                self.keep_.append(f)
        return self
    def transform(self, X):
        return pd.DataFrame(X, columns=FEATURES)[self.keep_].to_numpy()

results["Guerra-selected features (chosen inside each fold)"] = score(make_pipeline(GuerraSelect(), forest()), train[FEATURES], y)
results["prototype classifier (nearest class average)"] = score(make_pipeline(StandardScaler(), NearestCentroid()), train[FEATURES], y)

**Raw pixels + PCA** is the "let the computer find the features" option without a neural network: shrink each photo to 64 × 44 grey pixels (2,816 numbers) and squeeze those into 30 numbers with **PCA** (principal component analysis, which keeps the directions in which photos differ most). PCA is fitted inside each training fold so the test photos stay unseen.

In [ ]:
def pixels(job_ids, correct_light=True):
    lookup = photos.set_index("job_id")["file"]
    return np.stack([np.asarray(Image.fromarray(load_grey(lookup[j], correct_light)).resize((64, 44))).ravel() for j in job_ids])

X_pix = pixels(train["job_id"])
results["raw pixels + PCA (no hand features)"] = score(make_pipeline(StandardScaler(), PCA(30, random_state=42), forest()), X_pix, y)

### Optional: ResNet-18 features (run on your laptop)
A **ResNet-18** is a neural network already trained on 1.2 million everyday photos (ImageNet). We cut off its last layer and use the 512 numbers it produces for each photo as features, the same idea as Wu 2023. This needs PyTorch (`pip install torch torchvision`) and downloads the network weights (~45 MB) the first time. If PyTorch isn't installed, the cell says so and the ablation table simply has no ResNet row.

In [ ]:
try:
    import torch, torchvision
    weights = torchvision.models.ResNet18_Weights.DEFAULT
    net = torchvision.models.resnet18(weights=weights)
    net.fc = torch.nn.Identity()          # drop the ImageNet classifier, keep the 512 features
    net.eval()
    prep = weights.transforms()
    lookup = photos.set_index("job_id")["file"]
    with torch.no_grad():
        X_cnn = np.stack([net(prep(Image.open(lookup[j]).convert("RGB")).unsqueeze(0)).numpy()[0] for j in train["job_id"]])
    results["ResNet-18 features (pretrained, frozen)"] = score(forest(), X_cnn, y)
    results["ResNet-18 + hand features"] = score(forest(), np.hstack([X_cnn, train[FEATURES].to_numpy()]), y)
except ImportError:
    print("PyTorch not installed: skipping the ResNet-18 rows (pip install torch torchvision to add them)")

In [ ]:
ablation = pd.DataFrame(results).T
ablation["change_vs_full"] = ablation["macro_F1"] - full["macro_F1"]
ablation = ablation[["macro_F1", "F1_sd", "change_vs_full", "accuracy", "photos"]]
ablation.to_csv(OUT / "photo_ablation.csv")
display(ablation.style.format({"macro_F1": "{:.3f}", "F1_sd": "{:.3f}", "change_vs_full": "{:+.3f}", "accuracy": "{:.1%}", "photos": "{:.0f}"}))

plt.figure(figsize=(8, 0.35 * len(ablation) + 1))
order = ablation.sort_values("macro_F1")
plt.barh(order.index, order["macro_F1"], xerr=order["F1_sd"], color=["tab:red" if i.startswith("FULL") else "tab:blue" for i in order.index])
plt.axvline(full["macro_F1"], color="tab:red", ls="--", lw=1)
plt.xlabel("macro F1 (5-fold × 3 cross-validation)"); plt.xlim(0, 1.02); plt.title("Photo model ablation")
plt.tight_layout(); plt.savefig(FIG / "photo_ablation.png", dpi=150); plt.show()

### How to read the ablation
On my run (yours should be the same or very close, because every random step has a fixed `random_state`):
- **Context matters most.** "photo only (no tilt_off)" and "without geometry" drop macro F1 by about 0.07 to 0.08. Almost all of that drop is bevel_deviation: a correctly tilted 22.5° bevel and a wrongly tilted straight cut look alike unless you know the programmed angle. **This is a measured argument for the multimodal knowledge graph:** a photo becomes much more useful once it's linked to the job's settings, and that link is exactly what the graph provides.
- **No single feature group is enough.** "only ..." rows score 0.5 to 0.8, because each group only sees some defects. The groups complement each other.
- **A handful of features is enough.** The Guerra relevance/redundancy filter, re-run inside every training fold so the test photos never help choose the features, keeps about 4 features and scores about as well as all 20. In the report, the smaller set is easier to explain.
- **Cleaning barely changed the score here.** "no lighting correction" and "keep blurry photos" are within the ± of FULL. Say so honestly: the synthetic lighting changes are mild, and only 17 photos are blurry. Real phone photos are usually worse, so keep the steps.
- **Hand features beat generic ones on ~350 photos.** Raw pixels + PCA reached about 0.69 and the prototype classifier about 0.79. With few photos, telling the model *what to look at* beats letting it guess. ResNet-18 (run it on your laptop) is the fair test of that claim.
- **"only quality" is not near zero** (about 0.5), because `sharpness` also measures texture: rough edges have many sharp lines. Another feature measuring more than its name says.
- **The ± (F1_sd)** is how much the score moves between folds. Two rows whose difference is smaller than this are not really different.

## 9. Which features does the model rely on?
**Permutation importance:** shuffle one feature's column (break its link to the label) and see how much the score falls. A big fall means the model relied on it. We measure it on photos held out from training.

In [ ]:
Xtr, ytr = train[FEATURES], y
idx_train, idx_test = next(StratifiedKFold(5, shuffle=True, random_state=0).split(Xtr, ytr))
m = forest().fit(Xtr.iloc[idx_train], ytr.iloc[idx_train])
imp = permutation_importance(m, Xtr.iloc[idx_test], ytr.iloc[idx_test], scoring="f1_macro", n_repeats=20, random_state=42)
imp = pd.Series(imp.importances_mean, index=FEATURES).sort_values()
plt.figure(figsize=(7, 6)); plt.barh(imp.index, imp.values); plt.xlabel("drop in macro F1 when shuffled")
plt.title("Permutation importance (held-out photos)"); plt.tight_layout(); plt.savefig(FIG / "photo_importance.png", dpi=150); plt.show()

### Look at the mistakes
Numbers never tell the whole story. Here are the photos where the model and the inspector disagree. For each one, ask: **is the model wrong, or the inspector?** Inspectors make mistakes too, and a disagreement is a useful flag in a real factory.

In [ ]:
wrong = train[pred != y].assign(predicted=pred[pred != y])
print(len(wrong), "disagreements")
if len(wrong):
    n = min(len(wrong), 12)
    fig, axes = plt.subplots((n + 3) // 4, 4, figsize=(12, 2.3 * ((n + 3) // 4)), squeeze=False)
    lookup = photos.set_index("job_id")["file"]
    for ax, (_, r) in zip(axes.flat, wrong.head(n).iterrows()):
        ax.imshow(Image.open(lookup[r["job_id"]])); ax.set_title(f"{r['job_id']}\ninspector: {r['label']}  model: {r['predicted']}", fontsize=8)
    for ax in axes.flat: ax.axis("off")
    plt.tight_layout(); plt.savefig(FIG / "photo_disagreements.png", dpi=150); plt.show()
wrong[["job_id", "label", "predicted", "BevelAngle_deg", "tilt_off", "blob_count", "dark_band_height"]]

## 10. Photo evidence for the knowledge graph
For every photo we save the model's predicted defect and its **confidence** (the share of trees that voted for it).
- Photos with a label get an **out-of-fold** prediction (made by a model that didn't train on that photo), so the graph never contains a prediction that simply remembers the answer.
- Photos labelled `unknown` or `burnt_edge` get a prediction from a model trained on all labelled photos: a suggestion for the inspector.
- Blurry photos get `unreadable` and no prediction.

In Phase 2 each row becomes a `Photo` node linked to its `Job`, with a `SHOWS` edge to the predicted `Defect` carrying the confidence and the top feature values as evidence the LLM can cite.

In [ ]:
proba = cross_val_predict(forest(), train[FEATURES], y, cv=StratifiedKFold(5, shuffle=True, random_state=42), method="predict_proba")
classes_sorted = sorted(CLASSES)
ev = train[["job_id", "label"]].copy()
ev["photo_defect"] = [classes_sorted[i] for i in proba.argmax(axis=1)]
ev["confidence"] = proba.max(axis=1).round(2)
ev["how"] = "out-of-fold"

rest = feats[~feats["label"].isin(CLASSES) & ~feats["blurry"]].copy()
final = forest().fit(train[FEATURES], y)
pr = final.predict_proba(rest[FEATURES])
rest_ev = rest[["job_id", "label"]].assign(photo_defect=[final.classes_[i] for i in pr.argmax(axis=1)], confidence=pr.max(axis=1).round(2), how="suggestion (no usable label)")

blurry = feats[feats["blurry"]][["job_id", "label"]].assign(photo_defect="unreadable", confidence=np.nan, how="blurry: retake photo")
evidence = pd.concat([ev, rest_ev, blurry]).merge(feats[["job_id"] + FEATURES], on="job_id").sort_values("job_id")
evidence.to_csv(OUT / "photo_evidence.csv", index=False)
print("saved", len(evidence), "rows ->", OUT / "photo_evidence.csv")
print(evidence["how"].value_counts().to_dict())
evidence[evidence["how"].str.startswith("suggestion")][["job_id", "label", "photo_defect", "confidence"]]

## 11. What to write in the report (and what not to claim)
- **Method:** photo cleaning (name parsing, label matching, blur flag by Laplacian variance, flat-field lighting correction) → 20 interpretable features in 5 groups → random forest, scored by repeated stratified 5-fold cross-validation with macro F1 → ablation by feature group, cleaning step, context and method.
- **Main finding to look for:** the photo alone struggles with bevel_deviation, and adding the job's programmed bevel angle (`tilt_off`) fixes it. This is a concrete, measured reason to link photos to job events in the graph instead of using them on their own.
- **Big caveat:** these photos are **synthetic**. Our data generator draws each defect with a fixed rule (for example, dross as dark blobs on the bottom edge), and the hand features were designed after looking at those photos. Very high scores here are partly *by construction* and will not carry over to real phone photos. State this plainly, and treat real photos (ask Prof. May) as the real test.
- **Labels:** the model learns the inspector's label, so it can only be as good as the inspector. Disagreements (step 9) should be reported as "flags for review", not as model errors by default.